# B2-022-probabilistic-latent-models — Practice p07 — Solution

**Type:** constrained-coding · **Difficulty:** core · **Concepts:** multivariate-gaussian

*50 minutes.*  
**Set:** B  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import torch

SEED = 20260927
mu = torch.tensor([2.0, -1.0, 0.0], dtype=torch.float64)
sigma = torch.tensor([[1.0, 0.8, 0.0],
                      [0.8, 2.0, -0.5],
                      [0.0, -0.5, 0.5]], dtype=torch.float64)
sigma_not_spd = torch.tensor([[1.0, 2.0, 0.0], [2.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=torch.float64)
N_SAMPLES = 200_000
MEAN_ATOL = 0.02
COV_ATOL = 0.03

## Solution

If $z\sim N(0,I)$ and $\Sigma=AA^\top$, then $\mu+Az\sim N(\mu,AA^\top)=N(\mu,\Sigma)$ (Session 1 covariance rule).
Both the Cholesky factor $L$ and the spectral factor $S=Q\Lambda^{1/2}$ satisfy $AA^\top=\Sigma$, so both samplers have the right distribution, but they map the *same* $z$ to *different* points because $S=LR$ for a nontrivial orthogonal $R$.
Row-wise, the sample $\mu+Lz_0$ is written in batch form as `mu + z @ L.T`.

In [ ]:
def sample_gaussian(mu, sigma, n, generator):
    try:
        L = torch.linalg.cholesky(sigma)
    except RuntimeError as exc:   # torch.linalg.LinAlgError subclasses RuntimeError
        raise ValueError("sigma is not symmetric positive definite") from exc
    d = mu.shape[0]
    z = torch.randn(n, d, generator=generator, dtype=torch.float64)
    return mu + z @ L.T


def spectral_factor(sigma):
    evals, Q = torch.linalg.eigh(sigma)
    return Q @ torch.diag(evals.sqrt())


def fresh():
    return torch.Generator().manual_seed(SEED)


def moments(samples):
    mean = samples.mean(dim=0)
    centered = samples - mean
    cov = centered.T @ centered / samples.shape[0]   # 1/n covariance
    return mean, cov


L = torch.linalg.cholesky(sigma)
samples = sample_gaussian(mu, sigma, N_SAMPLES, fresh())
samples_again = sample_gaussian(mu, sigma, N_SAMPLES, fresh())
mean_hat, cov_hat = moments(samples)

z = torch.randn(N_SAMPLES, 3, generator=fresh(), dtype=torch.float64)
first_row_expected = mu + L @ z[0]

S = spectral_factor(sigma)
spectral_samples = mu + z @ S.T
mean_s, cov_s = moments(spectral_samples)

try:
    sample_gaussian(mu, sigma_not_spd, 10, fresh())
    not_spd_raises = False
except ValueError:
    not_spd_raises = True
print(samples[0], (mean_hat - mu).abs().max().item(), (cov_hat - sigma).abs().max().item())
print((mean_s - mu).abs().max().item(), (cov_s - sigma).abs().max().item())

**Why the tolerances differ.**
Probe 1 compares Monte Carlo averages with population values, and those averages carry sampling error of order $\sigma/\sqrt n$ (about $0.003$ per mean coordinate here), so its tolerances are set as a margin over that $n$-dependent error.
Probes 2–3 compare two deterministic computations from the same seeded generator, so they must agree exactly (up to floating-point rounding), and any difference signals a bug rather than randomness.

### Answer check

In [ ]:
assert samples.shape == (N_SAMPLES, 3) and samples.dtype == torch.float64
assert S.shape == (3, 3) and S.dtype == torch.float64
# Probe 1
assert (mean_hat - mu).abs().max().item() <= MEAN_ATOL
assert (cov_hat - sigma).abs().max().item() <= COV_ATOL
# Probe 2
assert torch.equal(samples, samples_again)
# Probe 3
assert torch.allclose(samples[0], first_row_expected, atol=1e-12, rtol=1e-12)
assert torch.allclose(samples[0], torch.tensor([2.101807, -2.067620, 0.211849], dtype=torch.float64), atol=1e-6, rtol=0)
# Probe 4
assert torch.allclose(S @ S.T, sigma, atol=1e-10, rtol=1e-10)
assert (mean_s - mu).abs().max().item() <= MEAN_ATOL
assert (cov_s - sigma).abs().max().item() <= COV_ATOL
assert not torch.allclose(spectral_samples, samples, atol=1e-3, rtol=0)
# Probe 5
assert not_spd_raises